In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report

# Ensure reproducibility across optimization states
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
%matplotlib inline

print("✅ Training and mitigation environment initialized.")

✅ Training and mitigation environment initialized.


In [2]:
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Ingest our validated Parquet files
train_df = pd.read_parquet(PROCESSED_DIR / "train.parquet")
test_df = pd.read_parquet(PROCESSED_DIR / "test.parquet")

print(f"📖 Train set loaded: {train_df.shape[0]:,} rows")
print(f"📖 Test set loaded: {test_df.shape[0]:,} rows")

📖 Train set loaded: 204,277 rows
📖 Test set loaded: 51,070 rows


In [ ]:
# Create our categorical and numerical maps based on the dataset features
# For this dataset, we map categorical attributes to clean dummy indicators
X_train = train_df.drop(columns=['Default'])
y_train = train_df['Default']
X_test = test_df.drop(columns=['Default'])
y_test = test_df['Default']

# Isolate the protected demographic attribute for fairness calculations
# 1 = Young Cohort (Protected), 0 = Mature Cohort (Baseline)
X_train['is_young'] = (X_train['Age'] < 30).astype(int)
X_test['is_young'] = (X_test['Age'] < 30).astype(int)

# Identify categorical features to dummy-encode dynamically
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
if categorical_cols:
    if 'LoanID' in categorical_cols:
        categorical_cols.remove('LoanID')
        X_train = X_train.drop(columns=['LoanID'])
        X_test = X_test.drop(columns=['LoanID'])

    X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
    X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)
    # Align columns to guarantee identical shapes
    X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print(f"📐 Extracted Feature Matrix Shape: {X_train.shape[1]} active training vectors.")

MemoryError: Unable to allocate 38.9 GiB for an array with shape (204277, 204277) and data type bool